# Chapter 7 — Planning

The ReAct loop decides one step at a time. That works, and it has a specific weakness:
the agent cannot tell you what it *intends* to do, only what it just did.

A plan is reviewable, auditable, and — crucially — repairable when a step fails.

| Pattern | Reacts to | Catches |
|---|---|---|
| **Replanning** | a step *failing* | broken plumbing |
| **Reflection** | a *conclusion* the evidence does not support | confident nonsense |



## Setup

This lab installs from **one** `requirements.txt`.

In [1]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


repo: /content/aegis


In [2]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.5/247.5 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 59.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.1/4.1 MB 69.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.6/222.6 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 695.4/695.4 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [3]:
!python tools/check_env.py --chapter 7

dependencies
  ok      langgraph              open-source agent track (StateGraph/END)
  ok      langchain-core         message and tool primitives
  ok      langchain-community    RAGAS dependency — see the pin note
  ok      google-adk             Google Cloud agent track (Agent, Workflow)
  ok      mcp                    tool discovery and hardening (Ch 3, 9, 11)
  ok      openai                 the default real-model tier
  ok      langchain-openai       wires OpenAI into RAGAS
  ok      ragas                  evaluation (Ch 10)
  ok      sacrebleu              required by RAGAS BleuScore
  ok      opentelemetry-sdk      tracing (Ch 10)
  ok      chromadb               vector store (Ch 6)
  ok      rank-bm25              sparse retrieval for hybrid search (Ch 6)
  ok      pytest                 the test suite

critical pin
  ok      langchain-community 0.3.29 (compatible with ragas)

model access
  absent  OPENAI_API_KEY not set
          Offline labs still run: AEGIS_MODEL=mock
  

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [4]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


model tier: mock


## Chain-of-thought: reasoning as an artifact

The value for agents is not that the model "thinks better." It is that the reasoning
becomes an **artifact you can inspect and disagree with**. A verdict with no visible
reasoning cannot be reviewed.

Note the limitation §7.2.3 names, visible below: chain-of-thought reasons about what the
model already believes. It cannot discover that the log source is down. That is why the
ReAct loop interleaves acting with reasoning.


In [5]:
import sys
sys.path.insert(0, "labs/chapter-07-planning")   # this chapter's source lives beside the notebook
from planning.reasoning import chain_of_thought

incident = {"id": "INC-7", "category": "brute_force",
            "user": "j.okafor", "src_ip": "203.0.113.42"}

reasoning = chain_of_thought(incident)
for step in reasoning["thought"]:
    print("  thought:", step)
print()
print("  conclusion:", reasoning["conclusion"])
print()
print("Nothing here touched a tool. The reasoning is about what the model already")
print("believes - it cannot discover that a log source is unavailable.")


  thought: The alert is categorised brute_force for user j.okafor from 203.0.113.42.
  thought: Brute force means repeated failed logins; if a success follows, the account may be taken over.
  thought: I do not yet know whether 203.0.113.42 is a known-bad address, or whether j.okafor is privileged.
  thought: The verdict depends on evidence I have not gathered: reputation, log history, and privilege.

  conclusion: Likely credential attack; confidence is low until reputation and log evidence are retrieved.

Nothing here touched a tool. The reasoning is about what the model already
believes - it cannot discover that a log source is unavailable.


## The ReAct loop: thought, action, observation

Interleaving is the fix for what chain-of-thought cannot do. Each turn produces an
**action** and an **observation**, and the next thought is informed by what actually
came back — not by what the model assumed.

This is Chapter 1's loop, unchanged. Its `trajectory` is the record of that interleaving,
and it is what a plan gives up in exchange for being reviewable up front.


In [6]:
# Chapter 1's loop, reproduced in this chapter's scratch/ so the lab is self-contained.
from scratch.triage_agent import run as react_run

react = react_run(verbose=False)

print("the ReAct trajectory - each action produced an observation:")
for action, observation in react["trajectory"]:
    print(f'  action: {action:16} observation: {observation[:44]}...')
print()
print("Contrast with a PLAN: ReAct decides one step at a time, so it adapts -")
print("but it cannot tell you in advance what it intends to do. A plan can.")


the ReAct trajectory - each action produced an observation:
  action: ip_reputation    observation: {"ip": "203.0.113.42", "score": 92, "verdict...
  action: get_user_context observation: {"user": "j.okafor", "role": "Finance Analys...
  action: final            observation: Investigation complete. See structured findi...

Contrast with a PLAN: ReAct decides one step at a time, so it adapts -
but it cannot tell you in advance what it intends to do. A plan can.


## Plan-and-solve, and dynamic replanning

The plan carries its own contingencies. Step one targets the one-hour log window — the
precise source. If that is unavailable the plan already knows what to do instead.

Crucially, the degradation is **recorded**. An investigation that quietly substituted a
worse source and reported the same confidence is lying by omission.


In [7]:
from planning.planner import make_plan, execute_plan

plan = make_plan(incident)

print("the plan, printed before anything runs (a plan you can print is a plan you can stop):")
for i, step in enumerate(plan, 1):
    print(f'  {i}. {step.name:30} fallback={"yes" if step.fallback else "no"}')

print()
degraded = execute_plan(make_plan(incident), unavailable_tools={"auth_fail_1h"})
for name, status, observation in degraded.executed:
    print(f'  {status:11} {name:30} {observation[:36]}...')

print()
print("replanned:", degraded.replanned)
print("The primary source was gone and Aegis still reached a verdict -")
print("by knowing, in advance, what a worse-but-workable substitute looked like.")


the plan, printed before anything runs (a plan you can print is a plan you can stop):
  1. correlate auth failures        fallback=yes
  2. check source IP reputation     fallback=no
  3. assess user privilege          fallback=no

  replanned   correlate auth failures        {"count": 4, "window": "24h", "resul...
  ok          check source IP reputation     {"ip": "203.0.113.42", "score": 92, ...
  ok          assess user privilege          {"user": "j.okafor", "role": "Financ...

replanned: True
The primary source was gone and Aegis still reached a verdict -
by knowing, in advance, what a worse-but-workable substitute looked like.


## Tool selection under ambiguity

People assume this decision is obvious. It is not: when two tools both look relevant,
picking confidently is how an agent invents work.

The **margin** between best and second-best is the signal — the same measured-dial idea
as Chapter 9's routing confidence. A confident pick at a 0.01 margin is a coin flip that
will look like a judgment in the trace.


In [8]:
from planning.reasoning import select_tool

GOALS = [
    "check whether this ip address is malicious",
    "find the auth failure events in the logs",
    "is this account privileged",
    "look up the history and reputation of this identity",
]

for goal in GOALS:
    choice = select_tool(goal)
    if choice["confident"]:
        print(f'{goal[:46]:48} -> {choice["tool"]:18} margin {choice["margin"]}')
    else:
        print(f'{goal[:46]:48} -> REFUSED: {choice["reason"]}')
        print(f'{"":48}    candidates: {choice.get("candidates")}')
print()
print("The last goal fits two tools almost equally. Refusing beats guessing.")


check whether this ip address is malicious       -> ip_reputation      margin 0.6
find the auth failure events in the logs         -> search_logs        margin 0.667
is this account privileged                       -> get_user_context   margin 0.667
look up the history and reputation of this ide   -> REFUSED: margin 0.0 below 0.15: get_user_context vs search_logs fit almost equally
                                                    candidates: ['get_user_context', 'search_logs']

The last goal fits two tools almost equally. Refusing beats guessing.


## Reflection: catching a conclusion the evidence does not support

Replanning reacts to a step *failing*. Reflection critiques a *conclusion*. One catches
broken plumbing; the other catches confident nonsense.

Reflection is not a retry. It is a **reviewer** — with the authority to overrule the
draft and the obligation to say why.


In [9]:
from planning.planner import summarize, reflect

# strip the evidence, then claim a compromise anyway
hollow = execute_plan(make_plan(incident))
hollow.executed = [(n, s, "") for n, s, _ in hollow.executed]

overclaim = {"steps": len(hollow.executed), "replanned": False,
             "reached_reputation": False, "verdict": "confirmed_compromise"}

final = reflect(hollow, overclaim)

print("draft verdict:", overclaim["verdict"])
print("final verdict:", final["verdict"])
print("revised:      ", final["revised"])
for note in final["reflection"]:
    print("  -", note)
print()
print("Nothing crashed. No exception, no monitor, no error budget entry.")
print("The agent was fluent, confident, and unsupported.")


draft verdict: confirmed_compromise
final verdict: inconclusive
revised:       True
  - draft claims compromise but no reputation source returned 'malicious' - downgrading to 'suspected'
  - fewer than two evidence-bearing observations - a verdict cannot rest on a single source; downgrading to 'inconclusive'

Nothing crashed. No exception, no monitor, no error budget entry.
The agent was fluent, confident, and unsupported.


### Both patterns, one run

If you can construct a case where both fire, you understand why both exist.


In [10]:
both = execute_plan(make_plan(incident), unavailable_tools={"auth_fail_1h"})
replanned = both.replanned                       # the world broke; the plan coped
both.executed = [(n, s, "") for n, s, _ in both.executed]   # and evidence is thin
verdict = reflect(both, {**summarize(both), "verdict": "confirmed_compromise"})

print("replanned (plumbing repaired):", replanned)
print("revised   (claim overruled):  ", verdict["revised"], "->", verdict["verdict"])
print()
print("Two different failures. Two different mechanisms. Both fired.")


replanned (plumbing repaired): True
revised   (claim overruled):   True -> inconclusive

Two different failures. Two different mechanisms. Both fired.


---

## What you built

A plan-and-solve agent with visible reasoning, contingencies carried in the plan, a tool
selector that refuses when two tools fit equally, and a reviewer that downgrades an
unsupported verdict.

- **A printed plan can be stopped before it runs.**
- **Record the degradation** — a silent fallback is a lie by omission.
- **Reflection catches the failure that throws no exception.**

**Next:** Chapter 8 splits Aegis into a team, and only one member may write to the world.
